# Simple Pure Python Ticket Solver

This notebook solves the **exact same support-ticket problem** using **100% Plain Python functions and `model.invoke()`**.

* **NO pipes (`|`)**
* **NO `RunnableParallel` or `RunnablePassthrough`**
* **NO `StateGraph` or graph compilation**
* **NO classes or Pydantic**

Just standard Python functions, dictionaries, and simple `model.invoke()` calls that every beginner understands immediately!

---

### The 3 Ways to Build AI Pipelines

| Approach | File | Best For |
| :--- | :--- | :--- |
| **1. Pure Python** | `Simple_Pure_Python_Ticket_Solver.ipynb` | Quick scripts, ultimate readability, zero framework lock-in |
| **2. LangChain LCEL** | `Simple_LangChain_Ticket_Solver_Core_Only.ipynb` | Declarative pipelines with pipes (`\|`), built-in streaming/batching |
| **3. LangGraph** | `Simple_LangGraph_Ticket_Solver.ipynb` | Complex state, loops, conditional routing, visual graph diagrams |


## 1. Install Dependencies

Notice we only need `langchain-openai` to call OpenAI (or `openai` SDK directly). No extra graph or chain libraries needed!


In [ ]:
!pip -q install -U langchain-openai python-dotenv


## 2. Add your OpenAI API key

The notebook automatically checks in this order:
1. **Local `.env` file** (in project root)
2. **Environment variable** (`OPENAI_API_KEY`)
3. **Google Colab Secrets** (click 🔑 Secrets on the left)
4. **Interactive prompt** (if none of the above are found)


In [ ]:
import os
import getpass

# 1. Automatically load .env file if available
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(), override=True)
except Exception:
    pass

# Pure Python fallback: read .env directly if python-dotenv isn't installed
if not os.environ.get("OPENAI_API_KEY") and os.path.exists(".env"):
    try:
        with open(".env", "r") as f:
            for line in f:
                line = line.strip()
                if line.startswith("OPENAI_API_KEY=") and not line.startswith("#"):
                    val = line.split("=", 1)[1].strip().strip('"').strip("'")
                    if val:
                        os.environ["OPENAI_API_KEY"] = val
                        break
    except Exception:
        pass


def read_openai_key():
    # Check if already loaded from .env or system environment
    key = os.environ.get("OPENAI_API_KEY")
    if key:
        return key

    # Check Google Colab Secrets
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key
    except Exception:
        pass

    # Last option: ask securely
    return getpass.getpass("Paste your OPENAI_API_KEY: ")


os.environ["OPENAI_API_KEY"] = read_openai_key()

print("✅ OpenAI API key is ready.")


## 3. Initialize Model

We initialize the model once. Every step simply calls `model.invoke("prompt string").content`!


In [ ]:
from langchain_openai import ChatOpenAI

# ---------------------------------------------------------
# MODEL
# ---------------------------------------------------------

model = ChatOpenAI(
    model="gpt-5.6-luna"
)

print("✅ Model ready.")


## 4. Step 1: Input Validation & Cleaning

A plain Python function that trims whitespace and validates input. No `RunnableLambda` needed!


In [ ]:
# ---------------------------------------------------------
# 1. CLEAN MESSAGE
# Plain Python string manipulation
# ---------------------------------------------------------

def clean_ticket(message: str) -> str:
    message = message.strip()

    if not message:
        raise ValueError("Ticket cannot be empty.")

    return message


## 5. Step 2: Analyse the Ticket

No `RunnableParallel`, no complex dictionary plumbing. Just 3 simple `model.invoke()` calls using regular Python f-strings!


In [ ]:
# ---------------------------------------------------------
# 2. ANALYSE TICKET
# Direct model.invoke() calls with regular f-strings
# ---------------------------------------------------------

def analyse_ticket(ticket: str) -> dict:
    summary = model.invoke(
        f"Summarize this support ticket in one short sentence:\n{ticket}"
    ).content.strip()

    category = model.invoke(
        f"Return only one category: billing, technical, or account.\nTicket: {ticket}"
    ).content.strip()

    urgency = model.invoke(
        f"Return only one urgency: low, medium, or high.\nTicket: {ticket}"
    ).content.strip()

    return {
        "summary": summary,
        "category": category,
        "urgency": urgency,
    }


## 6. Step 3: Make the Decision

Takes the ticket and analysis dictionary, formats a prompt, and asks the model for the decision.


In [ ]:
# ---------------------------------------------------------
# 3. MAKE DECISION
# Simple prompt + model.invoke()
# ---------------------------------------------------------

def make_decision(ticket: str, analysis: dict) -> str:
    prompt = f"""
Review this ticket and the initial analysis:

Ticket:
{ticket}

Analysis:
{analysis}

Give the final decision:
1. Should this ticket be escalated to a human? (Yes/No)
2. Brief reason why.
"""

    return model.invoke(prompt).content.strip()


## 7. Step 4: Write Customer Reply

Generates an empathetic customer reply from the ticket and decision.


In [ ]:
# ---------------------------------------------------------
# 4. CUSTOMER REPLY
# Simple prompt + model.invoke()
# ---------------------------------------------------------

def write_reply(ticket: str, decision: str) -> str:
    prompt = f"""
You are a customer-support assistant.

Customer message:
{ticket}

Decision:
{decision}

Write a short and helpful reply to the customer.
Do not mention prompts, AI, or internal system details.
"""

    return model.invoke(prompt).content.strip()


## 8. Step 5: The Main Workflow (`solve_ticket`)

Look at how clear and readable this is. Any Python developer—regardless of experience—can read this top-to-bottom without learning any framework!


In [ ]:
# ---------------------------------------------------------
# 5. THE WORKFLOW FUNCTION
# Call each function sequentially in plain Python
# ---------------------------------------------------------

def solve_ticket(message: str) -> dict:
    """
    Give this function one customer message.
    It calls each step in plain Python and prints the result.
    """

    # 1. Clean
    ticket = clean_ticket(message)

    # 2. Analyse
    analysis = analyse_ticket(ticket)

    # 3. Decide
    decision = make_decision(ticket, analysis)

    # 4. Reply
    reply = write_reply(ticket, decision)

    result = {
        "ticket": ticket,
        "analysis": analysis,
        "decision": decision,
        "reply": reply,
    }

    print("\n========== TICKET RESULT (Pure Python) ==========")
    print("Message  :", result["ticket"])
    print("\nAnalysis :")
    for key, val in result["analysis"].items():
        print(f"  - {key}: {val}")
    print("\nDecision :\n", result["decision"])
    print("\nReply to customer:\n", result["reply"])
    print("=================================================")

    return result


print("✅ Pure Python workflow ready.")
print("Now call: solve_ticket('your message')")


## 9. Run ONE function

Test the entire pure Python flow with a customer message!


In [ ]:
result = solve_ticket(
    "My account is locked and I have an important meeting in 10 minutes."
)


# 🎓 The Ultimate Comparison Guide

You now have 3 working notebooks for the exact same problem:

```text
lcel/
├── Simple_Pure_Python_Ticket_Solver.ipynb   ← Pure Python (Functions & Dicts)
├── Simple_LangChain_Ticket_Solver_Core_Only.ipynb ← LangChain LCEL (Pipes '|')
└── Simple_LangGraph_Ticket_Solver.ipynb      ← LangGraph (StateGraph & Nodes)
```

---

### Comparison Matrix

| Criteria | 1. Pure Python | 2. LangChain LCEL | 3. LangGraph |
| :--- | :--- | :--- | :--- |
| **Learning Curve** | Lowest (Standard Python) | Medium (New syntax `\|`, Runnables) | Medium-High (State, Graph, Nodes) |
| **Syntax** | `def func(x): return y` | `chain = prompt \| model \| parser` | `builder.add_node()`, `builder.add_edge()` |
| **State** | Plain dictionaries & variables | Hidden via `.assign()` | Explicit `TypedDict` blackboard |
| **Parallelism** | Manual (`ThreadPoolExecutor`) | Automatic (`RunnableParallel`) | Automatic in graph branch edges |
| **Visualization** | None | Tracing / Logs | Native Mermaid diagrams (`draw_mermaid_png()`) |
| **Best For** | Scripts, MVPs, beginner students | Simple linear production pipelines | Complex agents, loops, human-in-the-loop |

---

### Teaching Strategy for Students:
1. **Start with Pure Python**: Let students see that LLM apps are just Python functions calling `model.invoke()`.
2. **Introduce LCEL**: Show how pipes (`|`) eliminate boilerplate and enable automatic parallelism.
3. **Graduate to LangGraph**: Show how real-world multi-step systems with memory and branching are organized as visual graphs!
